# E045 Hierarchical Expert-Transformer RL — One Shot
目的: E043をbaseに、3日macro counterfactual datasetを作る前の再現・監査を一括実行する。Kaggleへ自動提出しない。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/kaggriculture")
EXP=ROOT/"03_experiments"/"E045_Hierarchical_Expert_Transformer_RL"
EXP.mkdir(parents=True,exist_ok=True)
print(EXP)


In [ ]:
# How: GitHub experiment branchを取得して同一コードで再現する。
!git clone -q -b research/e045-hierarchical-expert-transformer https://github.com/nagasora/kaggriculture.git /content/kaggriculture_e045
%cd /content/kaggriculture_e045
!python -m pip install -q -e . pytest torch scikit-learn


In [ ]:
# What: E045 safety invariants.
!pytest -q tests/test_e045_hierarchical.py


## Inputs
TOP5_BANK と E037派生データのパスだけ既存Drive構成に合わせて解決する。見つからなければ候補を表示して停止する。

In [ ]:
from pathlib import Path

def pick(pattern):
    xs=list(ROOT.glob(pattern))
    print(pattern, len(xs))
    for x in xs[:10]: print(" ",x)
    return xs[0] if xs else None

TOP5_BANK=pick("03_experiments/**/top5_plan_bank.json")
CAUSAL=pick("03_experiments/**/causal_chunks.npz")
SPLIT=pick("03_experiments/**/SPLIT_MANIFEST.json")
assert TOP5_BANK is not None, "top5_plan_bank.json not found"


In [ ]:
# How: 最新top trajectoryのmacro一貫性を再計算する。
OUT=EXP/"macro_bank_analysis.json"
!PYTHONPATH=src python scripts/e045_analyze_plan_bank.py --bank "$TOP5_BANK" --out "$OUT"


In [ ]:
# How: E037派生特徴が展開済みならfamily-classification probeを再現する。
if CAUSAL and SPLIT:
    OUT2=EXP/"representation_probe.json"
    !python scripts/e045_representation_probe.py --npz "$CAUSAL" --manifest "$SPLIT" --out "$OUT2" --epochs 20
else:
    print("E037 causal_chunks/SPLIT_MANIFEST not expanded. Macro analysis is complete; expand E037 archive before this optional probe.")


## Stage 2 entry point
次の学習対象はfamily分類ではなく Q(history, option)。learnerが実際に訪れた72手境界stateをforkし、互換expertのcounterfactual terminal returnを作る。PPOはこのdatasetでE043をfresh league上回った後だけ実行する。